In [1]:
# Import Library yang dibutuhkan
import rasterio
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from rasterio.features import shapes
from rasterio.mask import mask
import imblearn
import joblib
import scipy
from sktime.classification.sklearn import RotationForest
from xgboost import XGBClassifier, XGBRFClassifier
import xgboost as xgb
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.feature_selection import RFECV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score, log_loss, make_scorer
from imblearn.metrics import geometric_mean_score
import time

In [2]:
#importing the raster data
raster_file = 'C:\Master of Remote Sensing\Python Code\EL_Research\Imagery\Final_multisource_data_Landsat9.tif'
dataset = rasterio.open(raster_file)  
# Bands Names
desc = dataset.descriptions
print('Raster description: {desc}\n'.format(desc=desc)) 
#inspecting the raste file. The format is bands,row, column
L9_MS = dataset.read()
#tranpose the data into the form of row, column, bands
L9_MS = L9_MS.transpose(1, 2, 0)
L9_MS.shape
#adding the training data
sample = gpd.read_file('C:\Master of Remote Sensing\Python Code\Basic_ML_research\TD_New\TrainingSamples_rev21.shp')


Raster description: ('B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B10', 'NDVI', 'NDMI', 'EVI', 'MNDWI', 'MSAVI', 'EBBI', 'AWEI_sh', 'FlowA', 'TWI', 'TPILF', 'DEM', 'slope', 'aspect', 'TCB', 'TCG', 'TCW')



In [3]:
def extract_pixels_from_shapefile(shapefile, raster):
    training_samples = []
    for index, row in shapefile.iterrows():
        geometry = [row['geometry']]
        id_class = row['LUCID']

        # Mask raster untuk mendapatkan pixel values yang terdapat dalam geometry shapefile
        out_image, out_transform = mask(raster, geometry, crop=True)

        # out_image shape: (bands, height, width) -- Reshape menjadi (pixels, bands)
        out_image = out_image.reshape(raster.count, -1).T  # Sekarang (pixels, bands)

        # menghapus Nan Pixels (Jika ada piksel dalam suatu band bernilai NaN)
        valid_pixels = out_image[~np.isnan(out_image).any(axis=1)]

        # Menambahkan valid pixel sebagai feature bersamaan dengan kelas labelnya
        for pixel in valid_pixels:
            training_samples.append((pixel, id_class))

    # Converts the tuple list kedalam numpy array
    features = np.array([sample[0] for sample in training_samples])  # Extract pixel values (features)
    labels = np.array([sample[1] for sample in training_samples])    # Extract class labels
    return features, labels

In [4]:
# Menggunakan funtion extract pixel values yang sudah dibuat untuk mendapatkan fatures dan label:
features, labels = extract_pixels_from_shapefile(sample, dataset)

# Cek shape dari features dan labels
print(features.shape)
print(labels.shape)
# Membuat mask untuk cek Nan values dari feature dan label
nan_mask = np.isnan(features).any(axis=1) | np.isnan(labels)

# Apabila ada Nan value maka akan dihapuskan
features = features[~nan_mask]
labels = labels[~nan_mask]

# Menghitung final total sampel masing-masing kelas
sample_new = pd.DataFrame({'class': labels})
print(sample_new['class'].value_counts())

(21670, 24)
(21670,)
class
4.0     3476
19.0    2427
13.0    1928
10.0    1925
14.0    1811
17.0    1614
3.0     1269
8.0     1206
15.0    1136
0.0     1111
16.0    1099
18.0     574
7.0      510
2.0      445
6.0      261
11.0     229
1.0      152
5.0      151
9.0      134
12.0     118
20.0      94
Name: count, dtype: int64


In [5]:
# Split data: 80% for training and 20% for testing
x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.4, stratify=labels, random_state=42)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

# Calculate the number of pixels for each class in the training set
unique_classes_train, class_counts_train = np.unique(y_train, return_counts=True)

# Calculate the number of pixels for each class in the testing set
unique_classes_test, class_counts_test = np.unique(y_test, return_counts=True)

# Combine into a single DataFrame for easier visualization
class_distribution = pd.DataFrame({
    'Class': np.union1d(unique_classes_train, unique_classes_test),
    'Pixel_Count_Train': [class_counts_train[unique_classes_train.tolist().index(cls)] if cls in unique_classes_train else 0 for cls in np.union1d(unique_classes_train, unique_classes_test)],
    'Pixel_Count_Test': [class_counts_test[unique_classes_test.tolist().index(cls)] if cls in unique_classes_test else 0 for cls in np.union1d(unique_classes_train, unique_classes_test)]
})

# Display the combined class distribution
print("Class Distribution (Training and Testing Sets):")
print(class_distribution)
#class_distribution.to_csv('C:/Master of Remote Sensing/Python Code/HyperparameterTune_L9/train_test_EL.csv')

(13002, 24) (8668, 24) (13002,) (8668,)
Class Distribution (Training and Testing Sets):
    Class  Pixel_Count_Train  Pixel_Count_Test
0     0.0                667               444
1     1.0                 91                61
2     2.0                267               178
3     3.0                761               508
4     4.0               2086              1390
5     5.0                 91                60
6     6.0                157               104
7     7.0                306               204
8     8.0                724               482
9     9.0                 80                54
10   10.0               1155               770
11   11.0                137                92
12   12.0                 71                47
13   13.0               1157               771
14   14.0               1087               724
15   15.0                682               454
16   16.0                659               440
17   17.0                968               646
18   18.0          

In [8]:
np.save('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/X_train_original.npy', x_train)
np.save('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/X_test_original.npy', x_test)
np.save('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/Y_train_original.npy', y_train)
np.save('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/Y_test_original.npy', y_test)

In [ ]:
# Use RFE with cross-validation to  
# find the optimal number of features 
band_name = [
    'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B10', 
    'NDVI', 'NDMI', 'EVI', 'MNDWI', 'MSAVI', 'EBBI', 'AWEI_sh', 
     'FlowA', 'TWI', 'TPILF', 'DEM', 'slope', 'aspect', 
     'TCB', 'TCG', 'TCW']
# Conv
#  training data to a DataFrame
x_train_df = pd.DataFrame(x_train, columns=band_name)
x_test_df = pd.DataFrame(x_test, columns=band_name)
#Creating the recursive feature elimination
selector_rf = RFECV(estimator = RandomForestClassifier(n_estimators=400, random_state=42), cv=3, scoring='neg_log_loss') 
selector_rf.fit(x_train_df, y_train)
selector_xgb = RFECV(estimator=XGBClassifier(n_estimators=400, random_state=42), cv=3, scoring='neg_log_loss')
selector_xgb.fit(x_train_df, y_train)
selector_ert = RFECV(estimator=ExtraTreesClassifier(n_estimators=400, random_state=42), cv=3, scoring='neg_log_loss')
selector_ert.fit(x_train_df, y_train)
#print the optimal features
print("Optimal number of features for RF/Rotfor: %d" % selector_rf.n_features_) 
print("Optimal number of features for ERT: %d" % selector_ert.n_features_) 
# Get the mask of selected features
selected_mask_rf = selector_rf.support_
selected_mask_ert = selector_ert.support_
# Get the names of selected features
selected_bands_rf = x_train_df.columns[selected_mask_rf]
selected_bands_ert = x_train_df.columns[selected_mask_ert]

print('Selected bands for RF')
print(selected_bands_rf.tolist())
print('Selected bands for LGB')
print(selected_bands_ert.tolist())

x_train_xgb = selector_xgb.transform(x_train_df)
x_test_xgb = selector_xgb.transform(x_test_df)
# Get the mask of selected features
selected_mask_xgb = selector_xgb.support_
# Get the names of selected features
selected_bands_xgb = x_train_df.columns[selected_mask_xgb]
#Print the optimal bands name
print("Selected Bands for XGBrf:")
print(selected_bands_xgb.tolist())

x_train_rf = selector_rf.transform(x_train_df)
x_test_rf = selector_rf.transform(x_test_df)
x_train_ert = selector_ert.transform(x_train_df)
x_test_ert = selector_ert.transform(x_test_df)

joblib.dump(selector_rf, 'C:/Master of Remote Sensing/Python Code/RFECV_L9/RFECV_RF_Full.pkl',  compress=('zlib', 3))
joblib.dump(selector_ert, 'C:/Master of Remote Sensing/Python Code/RFECV_L9/RFECV_ERT_Full.pkl',  compress=('zlib', 3))

Optimal number of features for RF/Rotfor: 9
Optimal number of features for XGBRF: 22
Optimal number of features for ERT: 6
Selected Bands for XGBrf:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B10', 'NDVI', 'NDMI', 'EVI', 'MNDWI', 'MSAVI', 'EBBI', 'AWEI_sh', 'TPILF', 'DEM', 'slope', 'aspect', 'TCB', 'TCG', 'TCW']
Selected bands for RF
['B1', 'B4', 'B7', 'B10', 'EVI', 'MNDWI', 'DEM', 'TCG', 'TCW']
Selected bands for LGB
['B1', 'B3', 'B7', 'B10', 'DEM', 'TCG']


In [7]:
# Use RFE with cross-validation to  
# find the optimal number of features 
band_name = [
    'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B10', 
    'NDVI', 'NDMI', 'EVI', 'MNDWI', 'MSAVI', 'EBBI', 'AWEI_sh', 
     'FlowA', 'TWI', 'TPILF', 'DEM', 'slope', 'aspect', 
     'TCB', 'TCG', 'TCW']
# Conv
#  training data to a DataFrame
x_train_df = pd.DataFrame(x_train, columns=band_name)
x_test_df = pd.DataFrame(x_test, columns=band_name)
selector_rf = RFECV(estimator = RotationForest(n_estimators=300, random_state=42), cv=3, scoring='neg_log_loss') 
selector_rf.fit(x_train_df, y_train)


c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


ValueError: when `importance_getter=='auto'`, the underlying estimator RotationForest should have `coef_` or `feature_importances_` attribute. Either pass a fitted estimator to feature selector or call fit before calling transform.

In [6]:
selector_rf = joblib.load('C:/Master of Remote Sensing/Python Code/RFECV_L9/RFECV_RF_fixs.pkl')
selector_xgb = joblib.load('C:/Master of Remote Sensing/Python Code/RFECV_L9/RFECV_XGB_fixs.pkl')
selector_ert= joblib.load('C:/Master of Remote Sensing/Python Code/RFECV_L9/RFECV_ERT_Fixs.pkl')
selector_cb = joblib.load('C:/Master of Remote Sensing/Python Code/RFECV_L9/RFECV_CB_fixs.pkl')
band_name = [
    'B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B10', 
    'NDVI', 'NDMI', 'EVI', 'MNDWI', 'MSAVI', 'EBBI', 'AWEI_sh', 
     'FlowA', 'TWI', 'TPILF', 'DEM', 'slope', 'aspect', 
     'TCB', 'TCG', 'TCW']
# Conv
#  training data to a DataFrame
x_train_df = pd.DataFrame(x_train, columns=band_name)
x_test_df = pd.DataFrame(x_test, columns=band_name)
x_train_rf = selector_rf.transform(x_train_df)
x_test_rf = selector_rf.transform(x_test_df)
# Transform the data to include only selected features
x_train_xgb = selector_xgb.transform(x_train_df)
x_test_xgb = selector_xgb.transform(x_test_df)
x_train_ert = selector_ert.transform(x_train_df)
x_test_ert = selector_ert.transform(x_test_df)
x_train_cb = selector_cb.transform(x_train_df)
x_test_cb = selector_cb.transform(x_test_df)

In [8]:
selector_cb = RFECV(estimator= CatBoostClassifier(iterations = 400, random_state=42, 
        od_type='Iter',
        od_wait= 200), cv=3, scoring='accuracy')
selector_cb.fit(x_train_df, y_train)
selected_mask_cb = selector_cb.support_
# Get the names of selected features
selected_bands_cb = x_train_df.columns[selected_mask_cb]
#Print the optimal bands name
print("Selected Bands for CB:")
print(selected_bands_cb.tolist())
# Transform the data to include only selected features
x_train_cb = x_train_df.loc[:, selected_bands_cb]
#Transform the data for reducing the feature
x_train_reduce_cb = selector_cb.transform(x_train_df)
x_test_reduce_cb = selector_cb.transform(x_test_df)

Learning rate set to 0.191609
0:	learn: 2.3135848	total: 260ms	remaining: 1m 43s
1:	learn: 2.0926579	total: 324ms	remaining: 1m 4s
2:	learn: 1.9239020	total: 387ms	remaining: 51.2s
3:	learn: 1.8063606	total: 449ms	remaining: 44.4s
4:	learn: 1.7125335	total: 514ms	remaining: 40.6s
5:	learn: 1.6421157	total: 577ms	remaining: 37.9s
6:	learn: 1.5861402	total: 639ms	remaining: 35.9s
7:	learn: 1.5306832	total: 703ms	remaining: 34.4s
8:	learn: 1.4866115	total: 767ms	remaining: 33.3s
9:	learn: 1.4464491	total: 834ms	remaining: 32.5s
10:	learn: 1.4214383	total: 894ms	remaining: 31.6s
11:	learn: 1.3974862	total: 954ms	remaining: 30.8s
12:	learn: 1.3746714	total: 1.02s	remaining: 30.3s
13:	learn: 1.3558188	total: 1.08s	remaining: 29.8s
14:	learn: 1.3343382	total: 1.15s	remaining: 29.4s
15:	learn: 1.3180419	total: 1.21s	remaining: 29s
16:	learn: 1.3067257	total: 1.27s	remaining: 28.6s
17:	learn: 1.2956984	total: 1.33s	remaining: 28.3s
18:	learn: 1.2854272	total: 1.39s	remaining: 28s
19:	learn: 1.2

In [9]:
selector_xgb = RFECV(estimator=XGBClassifier(n_estimators=400, random_state=42), cv=3, scoring='neg_log_loss')
selector_xgb.fit(x_train_df, y_train)
x_train_xgb = selector_xgb.transform(x_train_df)
x_test_xgb = selector_xgb.transform(x_test_df)
# Get the mask of selected features
selected_mask_xgb = selector_xgb.support_
# Get the names of selected features
selected_bands_xgb = x_train_df.columns[selected_mask_xgb]
#Print the optimal bands name
print("Selected Bands for XGBrf:")
print(selected_bands_xgb.tolist())

Selected Bands for XGBrf:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B10', 'NDVI', 'NDMI', 'EVI', 'MNDWI', 'MSAVI', 'EBBI', 'AWEI_sh', 'FlowA', 'TWI', 'TPILF', 'DEM', 'slope', 'aspect', 'TCB', 'TCG', 'TCW']


In [12]:
print("Optimal number of features for CB: %d" % selector_cb.n_features_) 
joblib.dump(selector_xgb, 'C:/Master of Remote Sensing/Python Code/RFECV_L9/RFECV_XGB_fixs.pkl',  compress=('zlib', 3))

Optimal number of features for CB: 13


['C:/Master of Remote Sensing/Python Code/RFECV_L9/RFECV_XGB_fixs.pkl']

In [6]:
x_train_xgbrf = np.load('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/X_train_XGBRF.npy')
x_test_xgbrf = np.load('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/X_test_XGBRF.npy')
x_train_rf = np.load('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/x_train_rfrotfor60.npy')
x_test_rf = np.load('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/x_test_rfrotfor60.npy')
x_train_ert = np.load('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/x_train_ert_new.npy')
#x_test_ert = np.load ('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/x_test_ert_new.npy')
x_train_cb = np.load('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/x_train_cb_new.npy')
x_test_cb = np.load('C:/Master of Remote Sensing/Python Code/EL_Research/Training Data/numpy_TD/x_test_cb_new.npy')

In [8]:
#Parameter Grid for each model
#ERT
ERT_param = {
    'n_estimators' : [300, 500, 700, 900, 1100],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 7, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}
init_ET = ExtraTreesClassifier(random_state=42)
#Rotfor
rotfor_param_grid = {
    'n_estimators': [50, 100, 300, 500],
    'min_group': [1, 3, 5, 7],
    'max_group': [9, 13, 17],
}
rotformodel = RotationForest(random_state=42)
# Define the scoring metric
scoring_function = make_scorer(accuracy_score)
#Catboost
cb_param_grid = {
    'depth': [6, 9, 11],
    'l2_leaf_reg': [5, 7, 10, 13],
    'bagging_temperature': [0, 0.5, 0.7],
    'learning_rate': [ 0.001, 0.1, 1],
    'border_count': [64, 128, 254],
    'rsm': [0.1, 0.5, 1]
}

catboostmodel = CatBoostClassifier(iterations=600, random_state=42, od_type='Iter', od_wait=400, loss_function='MultiClass',
                                   eval_metric='MultiClass')

# Define the parameter grid for randomized search
xgb_param_grid = {
    'n_estimators': [500, 700, 900, 1100],
    'max_depth': [3, 6, 15, 20],
    'min_child_weight': [1, 3, 9, 13],
    'learning_rate': [0.001, 0.01, 0.1, 0.2],
    'subsample': [0.1, 0.5, 1.0],
    'colsample_bytree': [0.1, 0.5, 1.0]    
}
xgb_model = XGBClassifier(objective='multi:softprob', num_class = 21, eval_metric = 'mlogloss', random_state=42)
# Initialize a list to store results
results = []

In [28]:
# Randomized Search with Cross Validation for XGB
start_xgb = time.time()
# Initialize RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_param_grid,
    n_iter=20,  # Number of parameter settings to sample
    scoring='accuracy',  # Evaluation metric
    cv=7,  # 3-fold cross-validation
    verbose=2,  # Print progress
    random_state=42,
    n_jobs=-1  # Use all available cores
)
random_search.fit(x_train_xgb, y_train)
end_xgb = time.time()
time_xgb = end_xgb - start_xgb
results.append({
    'Classifier': 'XGB',
    'CV Accuracy': random_search.best_score_,
    'Best Params': random_search.best_params_,
    'Tuning Time': time_xgb
})
print('GCV best score for ET', random_search.best_score_)
print('GCV best parameter for ET', random_search.best_params_)

Fitting 7 folds for each of 20 candidates, totalling 140 fits
GCV best score for ET 0.7018141910560148
GCV best parameter for ET {'subsample': 1.0, 'n_estimators': 1100, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.2, 'colsample_bytree': 1.0}


In [17]:
start_ert = time.time()
et_rcv = RandomizedSearchCV(
    estimator = init_ET, 
    param_distributions= ERT_param, 
    n_iter=20, 
    cv=7, 
    verbose=2, 
    n_jobs=-1, 
    scoring='accuracy',
    random_state=42)
et_rcv.fit(x_train_ert, y_train)
end_ert = time.time()
time_ert = end_ert - start_ert
results.append({
    'Classifier': 'ERT',
    'Best Accuracy': et_rcv.best_score_,
    'Best Params': et_rcv.best_params_,
    'Tuning Time': time_ert
})
best_ert = et_rcv.best_estimator_
y_pred = best_ert.predict(x_test_ert)
y_prob = best_ert.predict_proba(x_test_ert)

print(log_loss(y_test, y_prob))
print(accuracy_score(y_test, y_pred))
print('GCV best score for ET', et_rcv.best_score_)
print('GCV best parameter for ET', et_rcv.best_params_)

Fitting 7 folds for each of 20 candidates, totalling 140 fits
0.9621082479959683
0.7080064605445316
GCV best score for ET 0.70358364073878
GCV best parameter for ET {'n_estimators': 300, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 20, 'bootstrap': False}


0.9361569810502477
0.7180433779418551
GCV best score for ET 0.7118387818791343
GCV best parameter for ET {'n_estimators': 1100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': None, 'bootstrap': False}

In [29]:
best_ert = et_rcv.best_estimator_
y_pred = best_ert.predict(x_test_ert)
y_prob = best_ert.predict_proba(x_test_ert)

print(log_loss(y_test, y_prob))
print(accuracy_score(y_test, y_pred))

#joblib.dump(best_ert, 'C:\Master of Remote Sensing\Python Code\EL_Research\Model Learnining\ERT_Best_estimator.pkl',  compress=('zlib', 3))
print('Time for ERT', time_ert)
best_xgb = random_search.best_estimator_
joblib.dump(best_xgb, 'C:\Master of Remote Sensing\Python Code\EL_Research\Model Learnining\XGB_Best_estimator.pkl',  compress=('zlib', 3))
print('TIME for XGBRF', time_xgb)

0.9621082479959683
0.7080064605445316
Time for ERT 55.80505585670471
TIME for XGBRF 725.5357611179352


n_estimators: 1100
max_depth: None
min_samples_split: 2
min_samples_leaf: 1
max_features: None
bootstrap: False

In [38]:
cb_start = time.time()
cb_rcv = RandomizedSearchCV(
    estimator= catboostmodel, 
    param_distributions=cb_param_grid, 
    n_iter=10,
    cv=7, 
    scoring='accuracy', 
    verbose=2, 
    n_jobs=-1)
cb_rcv.fit(x_train_reduce_cb, y_train)
cb_end = time.time()
print('Best score for CB RCV', cb_rcv.best_score_)
print('Best parameters for CB RCV', cb_rcv.best_params_)
cb_time = cb_end - cb_start
results.append({
    'Classifier': 'Catboost',
    'Best Accuracy': cb_rcv.best_score_,
    'Best Params': cb_rcv.best_params_,
    'Tuning Time': cb_time
})

Fitting 7 folds for each of 10 candidates, totalling 70 fits
0:	learn: 2.5974187	total: 279ms	remaining: 2m 47s
1:	learn: 2.3762704	total: 541ms	remaining: 2m 41s
2:	learn: 2.2181765	total: 810ms	remaining: 2m 41s
3:	learn: 2.0938057	total: 1.07s	remaining: 2m 40s
4:	learn: 1.9886532	total: 1.34s	remaining: 2m 39s
5:	learn: 1.9032454	total: 1.6s	remaining: 2m 38s
6:	learn: 1.8249544	total: 1.86s	remaining: 2m 37s
7:	learn: 1.7584873	total: 2.14s	remaining: 2m 38s
8:	learn: 1.7066744	total: 2.4s	remaining: 2m 37s
9:	learn: 1.6572294	total: 2.67s	remaining: 2m 37s
10:	learn: 1.6149400	total: 2.94s	remaining: 2m 37s
11:	learn: 1.5773607	total: 3.21s	remaining: 2m 37s
12:	learn: 1.5422392	total: 3.47s	remaining: 2m 36s
13:	learn: 1.5081917	total: 3.73s	remaining: 2m 36s
14:	learn: 1.4825189	total: 4.01s	remaining: 2m 36s
15:	learn: 1.4585882	total: 4.27s	remaining: 2m 35s
16:	learn: 1.4363298	total: 4.53s	remaining: 2m 35s
17:	learn: 1.4158596	total: 4.79s	remaining: 2m 35s
18:	learn: 1.39

In [39]:
# Compute log loss for LGBM
best_cb = cb_rcv.best_estimator_
cb_prob = best_cb.predict_proba(x_test_reduce_cb)
y_pred_cb = best_cb.predict(x_test_reduce_cb)
cb_log_loss = log_loss(y_test, cb_prob)
print(accuracy_score(y_test, y_pred_cb))
print(cb_log_loss)
print(cb_time)

0.7071988924780803
0.9304828231463245
3507.9401819705963


3507.9401819705963 + 7 min 36 sec (time for catboost)

In [40]:
modcb = CatBoostClassifier(iterations=1400, depth=9, bagging_temperature = 0.5, border_count=254, l2_leaf_reg=5, learning_rate=0.1, rsm=1,
                           od_type='Iter', od_wait=900, eval_metric='MultiClass')
modcb.fit(x_train_reduce_cb, y_train)
y_pred_cb_mod = modcb.predict(x_test_reduce_cb)
cb_prob_mod = modcb.predict_proba(x_test_reduce_cb)
cb_log_loss = log_loss(y_test, cb_prob_mod)
print(accuracy_score(y_test, y_pred_cb_mod))
print(cb_log_loss)

0:	learn: 2.6719863	total: 279ms	remaining: 6m 30s
1:	learn: 2.4232495	total: 542ms	remaining: 6m 18s
2:	learn: 2.2393834	total: 807ms	remaining: 6m 15s
3:	learn: 2.1082822	total: 1.06s	remaining: 6m 11s
4:	learn: 1.9979739	total: 1.33s	remaining: 6m 10s
5:	learn: 1.9107368	total: 1.59s	remaining: 6m 9s
6:	learn: 1.8361006	total: 1.86s	remaining: 6m 9s
7:	learn: 1.7733558	total: 2.13s	remaining: 6m 10s
8:	learn: 1.7159089	total: 2.38s	remaining: 6m 8s
9:	learn: 1.6688460	total: 2.64s	remaining: 6m 7s
10:	learn: 1.6257301	total: 2.9s	remaining: 6m 5s
11:	learn: 1.5865767	total: 3.16s	remaining: 6m 5s
12:	learn: 1.5475264	total: 3.46s	remaining: 6m 8s
13:	learn: 1.5152583	total: 3.73s	remaining: 6m 9s
14:	learn: 1.4872601	total: 4s	remaining: 6m 9s
15:	learn: 1.4607612	total: 4.27s	remaining: 6m 9s
16:	learn: 1.4367087	total: 4.54s	remaining: 6m 9s
17:	learn: 1.4155766	total: 4.79s	remaining: 6m 8s
18:	learn: 1.3943358	total: 5.05s	remaining: 6m 7s
19:	learn: 1.3774726	total: 5.32s	remai

In [41]:
joblib.dump(cb_rcv, 'C:\Master of Remote Sensing\Python Code\EL_Research\Model Learnining\CB_Best_estimator.pkl', compress=('zlib', 3))
joblib.dump(modcb, 'C:\Master of Remote Sensing\Python Code\EL_Research\Model Learnining\MODCB_RCV.pkl',compress=('zlib', 3) )

['C:\\Master of Remote Sensing\\Python Code\\EL_Research\\Model Learnining\\MODCB_RCV.pkl']

In [31]:
resultdf = pd.DataFrame(results)
print(resultdf)

  Classifier  Best Accuracy  \
0     Rotfor        0.69643   
1        XGB            NaN   

                                         Best Params  Tuning Time  CV Accuracy  
0  {'n_estimators': 500, 'min_group': 1, 'max_gro...  1079.425987          NaN  
1  {'subsample': 1.0, 'n_estimators': 1100, 'min_...   725.535761     0.701814  


In [ ]:
results = []
start_rotfor = time.time()
rotfor_rcv = RandomizedSearchCV(estimator=rotformodel, 
                                param_distributions=rotfor_param_grid, 
                                n_iter=10, 
                                cv=5, 
                                random_state=42, 
                                n_jobs=-1, 
                                verbose=3, 
                                scoring=scoring_function)
rotfor_rcv.fit(x_train, y_train)
end_rotfor = time.time()
time_rotfor = end_rotfor - start_rotfor
results.append({
    'Classifier': 'Rotfor',
    'Best Accuracy': rotfor_rcv.best_score_,
    'Best Params': rotfor_rcv.best_params_,
    'Tuning Time': time_rotfor
})
print('Rotfor best score', rotfor_rcv.best_score_)
print('rotfor best parameters', rotfor_rcv.best_params_)

Fitting 7 folds for each of 10 candidates, totalling 70 fits


c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Rotfor best score 0.6964302793516203
rotfor best parameters {'n_estimators': 500, 'min_group': 1, 'max_group': 13}


In [13]:
mod_rotfor = RotationForest(n_estimators=300, min_group=1, max_group=9)
mod_rotfor.fit(x_train, y_train)
y_rf = mod_rotfor.predict(x_test)


c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force

In [14]:
y_rf_prob = mod_rotfor.predict_proba(x_test)
print(accuracy_score(y_test, y_rf))
print(f1_score(y_test, y_rf, average='weighted'))
print(log_loss(y_test, y_rf_prob))
print(geometric_mean_score(y_test, y_rf, average='weighted'))

c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


0.7090447623442547
0.7399021523849142
0.9694694057479761
0.8194433490397939


In [26]:
best_rotfor = rotfor_rcv.best_estimator_
joblib.dump(best_rotfor, 'C:\Master of Remote Sensing\Python Code\EL_Research\Model Learnining\Rotfor_best_estimator.pkl', compress=('zlib', 3))
print(time_rotfor)

1079.425987482071


In [8]:
from sklearn.utils.class_weight import compute_class_weight
# Load models
model_xgb = joblib.load('C:\Master of Remote Sensing\Python Code\EL_Research\Model Learnining\XGB_Best_estimator.pkl')
model_ert = joblib.load('C:\Master of Remote Sensing\Python Code\EL_Research\Model Learnining\ERT_Best_estimator.pkl')
model_rotfor = joblib.load('C:\Master of Remote Sensing\Python Code\EL_Research\Model Learnining\Rotfor_best_estimator.pkl')
model_cb = joblib.load('C:\Master of Remote Sensing\Python Code\EL_Research\Model Learnining\MODCB_RCV.pkl')
# Classifiers and corresponding transformed test data
classifiers = {
    'XGB': {'model': model_xgb, 'x_train': x_train_xgb, 'x_test': x_test_xgb},
    'ERT': {'model': model_ert, 'x_train': x_train_ert, 'x_test': x_test_ert},
    'Rotfor': {'model': model_rotfor, 'x_train': x_train_rf, 'x_test': x_test_rf},
    'CB': {'model': model_cb, 'x_train': x_train_cb, 'x_test': x_test_cb},
}

# Store results
individual_accuracies = []

# Calculate class weights and evaluate classifiers
for name, clf_data in classifiers.items():
    clf = clf_data['model']
    x_train = clf_data['x_train']
    x_test = clf_data['x_test']

    # Calculate class weights based on RFECV-reduced training data
    unique_classes = np.unique(y_train)  # Assuming y_train is the same for all classifiers
    weights = compute_class_weight(
        class_weight='balanced',
        classes=unique_classes,
        y=y_train
    )
    class_weight_dict = dict(zip(unique_classes, weights))
    print(f"Class weights for {name}: {class_weight_dict}")

    # Predict using the current classifier
    y_pred = clf.predict(x_test)
    y_pred_proba = clf.predict_proba(x_test)

    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', sample_weight=[class_weight_dict[cls] for cls in y_test])
    w_logloss = log_loss(y_test, y_pred_proba, sample_weight=[class_weight_dict[cls] for cls in y_test])
    gmean = geometric_mean_score(y_test, y_pred, average='weighted', sample_weight=[class_weight_dict[cls] for cls in y_test])

    # Append results
    individual_accuracies.append((name, acc, f1, w_logloss, gmean))
    print(f'Accuracy of {name}: {acc:.4f}')
    print(f'F1 Score of {name}: {f1:.4f}')
    print(f'Log loss of {name}: {w_logloss:.4f}')
    print(f'Geometric Mean of {name}: {gmean:.4f}')

# Identify the worst-performing classifier
worst_classifier = min(individual_accuracies, key=lambda x: x[1])
print(f'\nWorst-performing classifier: {worst_classifier[0]} with accuracy: {worst_classifier[1]:.4f}')
print(f'F1 Score of worst-performing classifier: {worst_classifier[2]:.4f}')

# Convert results to DataFrame for easier visualization
columns = ['Classifier', 'Overall Accuracy', 'F1 Score', 'Log Loss', 'Geometric Mean']
result_df = pd.DataFrame(individual_accuracies, columns=columns)

# Save the DataFrame
result_df.to_csv('C:\Master of Remote Sensing\Python Code\EL_Research\Analysis_EL\ELModel_eval.csv', index=False)
print(result_df)

Class weights for XGB: {0.0: 0.9282501606339687, 1.0: 6.803767660910518, 2.0: 2.3188871054039595, 3.0: 0.8135911394781303, 4.0: 0.29680865634844544, 5.0: 6.803767660910518, 6.0: 3.943585077343039, 7.0: 2.023342670401494, 8.0: 0.8551696921862668, 9.0: 7.739285714285714, 10.0: 0.5360544217687074, 11.0: 4.519290928050052, 12.0: 8.720321931589536, 13.0: 0.5351277935547598, 14.0: 0.5695886450256276, 15.0: 0.9078341013824884, 16.0: 0.9395187513548667, 17.0: 0.6396103896103896, 18.0: 1.7998338870431894, 19.0: 0.4252354788069074, 20.0: 11.056122448979592}
Accuracy of XGB: 0.7087
F1 Score of XGB: 0.7443
Log loss of XGB: 1.2003
Geometric Mean of XGB: 0.8121
Class weights for ERT: {0.0: 0.9282501606339687, 1.0: 6.803767660910518, 2.0: 2.3188871054039595, 3.0: 0.8135911394781303, 4.0: 0.29680865634844544, 5.0: 6.803767660910518, 6.0: 3.943585077343039, 7.0: 2.023342670401494, 8.0: 0.8551696921862668, 9.0: 7.739285714285714, 10.0: 0.5360544217687074, 11.0: 4.519290928050052, 12.0: 8.720321931589536

c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force

Accuracy of Rotfor: 0.7043
F1 Score of Rotfor: 0.7401
Log loss of Rotfor: 1.2294
Geometric Mean of Rotfor: 0.8104
Class weights for CB: {0.0: 0.9282501606339687, 1.0: 6.803767660910518, 2.0: 2.3188871054039595, 3.0: 0.8135911394781303, 4.0: 0.29680865634844544, 5.0: 6.803767660910518, 6.0: 3.943585077343039, 7.0: 2.023342670401494, 8.0: 0.8551696921862668, 9.0: 7.739285714285714, 10.0: 0.5360544217687074, 11.0: 4.519290928050052, 12.0: 8.720321931589536, 13.0: 0.5351277935547598, 14.0: 0.5695886450256276, 15.0: 0.9078341013824884, 16.0: 0.9395187513548667, 17.0: 0.6396103896103896, 18.0: 1.7998338870431894, 19.0: 0.4252354788069074, 20.0: 11.056122448979592}
Accuracy of CB: 0.7108
F1 Score of CB: 0.7488
Log loss of CB: 1.1732
Geometric Mean of CB: 0.8145

Worst-performing classifier: Rotfor with accuracy: 0.7043
F1 Score of worst-performing classifier: 0.7401
  Classifier  Overall Accuracy  F1 Score  Log Loss  Geometric Mean
0        XGB          0.708699  0.744293  1.200270        0.8

In [44]:
def classify_raster_with_selected_features(raster, model, selected_features):
    # Read raster data
    raster_data = raster.read()  # Shape: (bands, height, width)

    # Extract only the selected features (bands)
    selected_data = raster_data[selected_features, :, :]  # Filter bands

    # Reshape raster data into (pixels, features)
    reshaped_data = selected_data.reshape(len(selected_features), -1).T  # (pixels, features)

    # Handle any invalid or missing data in the raster (e.g., NaN values)
    mask = np.isnan(reshaped_data).any(axis=1)  # Identify invalid pixels
    classified = np.full(reshaped_data.shape[0], -1, dtype=int)  # Initialize with a NoData value

    # Perform classification on valid pixels only
    classified[~mask] = model.predict(reshaped_data[~mask])

    # Reshape back to raster dimensions
    return classified.reshape((raster.height, raster.width))
# Save classified raster to disk
def save_raster(output_path, reference_raster, classified_data):
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=reference_raster.height,
        width=reference_raster.width,
        count=1,
        dtype=classified_data.dtype,
        crs=reference_raster.crs,
        transform=reference_raster.transform,
    ) as dst:
        dst.write(classified_data, 1)

In [45]:
selected_features_xgbrf = np.where(selector_xgb.support_)[0]  # Get indices of selected features (0-indexed)
selected_features_ert = np.where(selector_ert.support_)[0]
selected_features_cb = np.where(selector_cb.support_)[0]
# Perform classification with selected features
xgbrfmap = classify_raster_with_selected_features(dataset, model_xgb, selected_features_xgbrf)
ertmap = classify_raster_with_selected_features(dataset, model_ert, selected_features_ert)
#cbmap = classify_raster_with_selected_features(dataset, model_cb, selected_features_cb)
#cbmap = classify_raster_with_selected_features(raster, modcb, selected_features)

In [46]:
save_raster('C:/Master of Remote Sensing/Python Code/EL_Research/NewOutput/XGB_Final.tif', dataset, xgbrfmap)
save_raster('C:/Master of Remote Sensing/Python Code/EL_Research/NewOutput/ERT_Final.tif', dataset, ertmap)

In [49]:
# Get selected feature indices using the `support_` attribute
selected_band_indices = np.where(selector_rf.support_)[0]  # Get indices of selected features (bands)
def classify_raster_in_chunks(raster_path, model, selected_band_indices, chunk_size=128):
    # Open the raster file
    with rasterio.open(raster_path) as src:
        # Create an empty array to store classified data
        classified_data = np.full((src.height, src.width), -1, dtype=np.int32)  # NoData as -1

        # Read the raster data chunk by chunk
        for y in range(0, src.height, chunk_size):
            for x in range(0, src.width, chunk_size):
                # Define the window
                window_height = min(chunk_size, src.height - y)  # Handle edge case
                window_width = min(chunk_size, src.width - x)   # Handle edge case
                window = rasterio.windows.Window(x, y, window_width, window_height)

                # Read the data for this window
                data = src.read(window=window)
                data_reshaped = data.reshape((data.shape[0], -1)).T  # (pixels, bands)

                # Select only the required bands based on selected indices
                selected_data = data_reshaped[:, selected_band_indices]

                # Mask invalid data (e.g., NaN or all zeros)
                mask = np.isnan(selected_data).any(axis=1)
                valid_data = selected_data[~mask]

                # Classify valid data
                if valid_data.size > 0:
                    predicted = model.predict(valid_data)
                    # Fill classified data into the output array
                    classified_chunk = np.full(selected_data.shape[0], -1, dtype=np.int32)
                    classified_chunk[~mask] = predicted
                    classified_chunk = classified_chunk.reshape((window_height, window_width))

                    # Write the chunk to the classified_data array
                    classified_data[
                        y : y + window_height, x : x + window_width
                    ] = classified_chunk

    return classified_data

# Perform classification
classified_map = classify_raster_in_chunks(raster_file, model_rotfor, selected_band_indices, chunk_size=128)

c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\62853\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force

In [51]:
save_raster('C:/Master of Remote Sensing/Python Code/EL_Research/NewOutput/Rotfor_final.tif', dataset, classified_map)

In [50]:
# Get selected feature indices using the `support_` attribute
selected_band_indices = np.where(selector_cb.support_)[0]  # Get indices of selected features (bands)

output_path = 'C:/Master of Remote Sensing/Python Code/EL_Research/NewOutput/CB_final.tif'

with rasterio.open(raster_file) as src:
    # Read all bands into an array
    raster_data = src.read().astype("float32")  # Shape: (bands, rows, cols)
    profile = src.profile  # Save profile for saving the output raster

    # Select only the bands chosen by RFECV
    selected_bands_data = raster_data[selected_band_indices, :, :]  # Shape: (selected_bands, rows, cols)

    # Reshape raster data for prediction
    rows, cols = selected_bands_data.shape[1], selected_bands_data.shape[2]
    raster_data_reshaped = selected_bands_data.reshape(selected_bands_data.shape[0], -1).T  # Shape: (rows*cols, selected_bands)

# Predict land cover class for each pixel using the trained CatBoost model
predicted_classes = modcb.predict(raster_data_reshaped)

# Reshape predictions back to raster dimensions
predicted_classes_raster = predicted_classes.reshape(rows, cols)

# Save the classified raster
profile.update(dtype=rasterio.uint8, count=1)  # Update profile for single-band output
with rasterio.open(output_path, "w", **profile) as dst:
    dst.write(predicted_classes_raster.astype(rasterio.uint8), 1)
